In [ ]:
#Część 1

#(base) jovyan@jupyter115:~/notebooks$ export PATH=/home/jovyan/kafka/bin:$PATH
#(base) jovyan@jupyter115:~/notebooks$ kafka-topics.sh --create \
#  --topic transactions \
#  --bootstrap-server broker:9092

#kafka-topics.sh --list --bootstrap-server broker:9092
#Error while executing topic command : Topic 'transactions' already exists.
#[2026-04-12 16:24:58,062] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'transactions' already exists.
# (org.apache.kafka.tools.TopicCommand)

#WYNIK
__consumer_offsets
transactions


In [ ]:
#Część 2

#KOD

%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    tx_id = f"TX{random.randint(1, 9999):04d}"
    user_id = f"u{random.randint(1, 20):02d}"
    amount = round(random.uniform(5.0, 5000.0), 2)
    store = random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"])
    category = random.choice(["elektronika", "odzież", "żywność", "książki"])
    timestamp = datetime.now().isoformat()

    return {
        "tx_id": tx_id,
        "user_id": user_id,
        "amount": amount,
        "store": store,
        "category": category,
        "timestamp": timestamp
    }

#PĘTLA

while True:
    transaction = generate_transaction()
    producer.send('transactions', transaction)
    print(transaction)
    time.sleep(1)


In [ ]:
#Część 3

#Zad.3.1

#KOD

%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    transaction = message.value
    if transaction["amount"] > 3000:
        print(
            f"ALERT: {transaction['tx_id']} | "
            f"{transaction['amount']} PLN | "
            f"{transaction['store']} | "
            f"{transaction['category']}"
        )

#Zad.3.2

#KOD

%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję i dodaję risk_level...")

for message in consumer:
    transaction = message.value

    if transaction["amount"] > 3000:
        transaction["risk_level"] = "HIGH"
    elif transaction["amount"] > 1000:
        transaction["risk_level"] = "MEDIUM"
    else:
        transaction["risk_level"] = "LOW"

    print(transaction)


In [ ]:
#Część 4

#Zad.4.1

#KOD

%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    transaction = message.value
    store = transaction["store"]
    amount = transaction["amount"]

    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0) + amount
    msg_count += 1

    if msg_count % 10 == 0:
        print("\nSklep | Liczba | Suma | Średnia")
        print("-" * 40)
        for store_name in store_counts:
            count = store_counts[store_name]
            total = total_amount[store_name]
            avg = total / count
            print(f"{store_name} | {count} | {total:.2f} | {avg:.2f}")


#Zad. 4.2

#KOD

%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "sum": 0.0,
    "min": float("inf"),
    "max": float("-inf")
})

msg_count = 0

for message in consumer:
    transaction = message.value
    category = transaction["category"]
    amount = transaction["amount"]

    stats[category]["count"] += 1
    stats[category]["sum"] += amount
    stats[category]["min"] = min(stats[category]["min"], amount)
    stats[category]["max"] = max(stats[category]["max"], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\nKategoria | Liczba | Przychód | Min | Max")
        print("-" * 55)
        for cat, data in stats.items():
            print(
                f"{cat} | {data['count']} | {data['sum']:.2f} | "
                f"{data['min']:.2f} | {data['max']:.2f}"
            )


In [ ]:
#Część 5

#Pytania

#1: Po zatrzymaniu producenta konsument nie dostaje nowych wiadomości. Może dalej działać, ale nic nowego nie wypisze, dopóki producer nie zostanie uruchomiony ponownie.
#2: Jeśli dwóch konsumentów ma tę samą group_id, Kafka traktuje ich jako jedną grupę i rozdziela między nich wiadomości. Każdy konsument dostaje tylko część danych.
#3: Przetwarzanie bezstanowe nie pamięta poprzednich wiadomości i analizuje każdą osobno. Przetwarzanie stanowe przechowuje stan między wiadomościami, np. liczniki i sumy, dzięki czemu możliwe są agregacje.


In [ ]:
#Praca domowa

#Kod

%%file consumer_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='anomaly-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_transactions = defaultdict(list)

print("Nasłuchuję anomalii prędkości...")

for message in consumer:
    transaction = message.value
    user_id = transaction["user_id"]
    tx_time = datetime.fromisoformat(transaction["timestamp"])

    user_transactions[user_id].append(tx_time)

    cutoff = tx_time - timedelta(seconds=60)
    user_transactions[user_id] = [
        t for t in user_transactions[user_id] if t >= cutoff
    ]

    if len(user_transactions[user_id]) > 3:
        print(
            f"ALERT: użytkownik {user_id} wykonał "
            f"{len(user_transactions[user_id])} transakcje w ciągu 60 sekund"
        )
